Description: 
given the target configuration in joint space, finding the movement trajectory.


In [1]:
import torch, time
import sys, os
sys.path.append('./../')

import numpy as np
from panda_robot.config import absjoin, PANDA_XML, SHELF_URDF, BASE_URDF, \
    BASE_URDF

import sys, os


from panda_robot.envs.env_panda_shelf_gs import PandaShelf

from src.ttts_augment import TTTS
from robot_utils import Point2PointMotion, PlanarManipulatorCost


%load_ext autoreload
%autoreload 2

device = 'cuda' if torch.cuda.is_available() else 'cpu'

[I 07/03/25 10:42:02.769 62425] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


In [2]:
ARM_BASE_HEIGHT = 0.7

MAX_BATCH = 1000

In [3]:
def env_infos_gen():
    """
    This function generates the environment information 
    useful for defining the simulation environment.
    """
    shelf_info = {
        "name": "shelf",
        "urdf": SHELF_URDF,
        "base_pose": [0.7, 0.0, 0.015, 0, 0, 90],
    }
    print("SHELF_URDF: ", SHELF_URDF)

    base_info = {
        "name": "base",
        "urdf": BASE_URDF,
        "base_pose": [0.0, 0.0, 0.35, 0, 0, 90]
    }

    robot_info = {
        "name": "panda",
        "xml": PANDA_XML,
        "base_pose": [0.0, 0.0, 0.8, 0, 0, 0],
        # "conf": [0.0, -np.pi/4, 0.0, -np.pi/2, 
        #          0, np.pi/2, np.pi/4, 0.06, 0.06],
        "conf": [1.76, -4.5e-01, -1.77, -1.35,  4.9e-01,
                2.36, -2.8e-01, 0.06, 0.06],
        "jnt_names": ["joint1", "joint2", "joint3",
                    "joint4", "joint5", "joint6",
                    "joint7", "finger_joint1", "finger_joint2", ],
        "kp": np.array([4500, 4500, 3500, 3500, 2000, 2000, 
                            2000, 100, 100]),
        "kv": np.array([450, 450, 350, 350, 200, 200, 200, 
                        10, 10]),
        "force_range": {
            "lower": np.array([-87, -87, -87, -87, -12, -12, 
                                -12, -100, -100]),
            "upper": np.array([87,  87,  87, 87,  12,  12, 
                                12,  100,  100]),
        },
    }

    goal_info = {
        "goal_config": [ 4.3105, -0.7813, -4.4040, -2.4188, -0.5002,  4.6439, -1.9318,  0.0000,
          0.0000]
        # [2.6170, -2.1647, -1.3917, -2.0655,  2.4773,  0.7699, -1.7667,  0.0000,
        #   0.0000]
    }

    

    utilities = {
        "target_x1": [0.70, -0.15, 1.5],
        "target_x2": [0.60, 0.15, 1.1],
        # "1": [0.60, 0.15, 1.5],
        # "2": [0.55, -0.2, 1.6],
        # "3": [0.5, -0.2, 1.2],
        # "4": [0.6, 0.15, 1.1],
        # "5": [0.7, 0.1, 1.1],
        # "6": [0.6, -0.1, 1.1],
    }

    env_infos = {
        "shelf": shelf_info,
        "base": base_info,
        "robot": robot_info,
        "utilities": utilities,
        "goal_info": goal_info,
    }

    return env_infos

def test_collision(env_infos):
    action1 = [[0.0, -np.pi/4, 0.0, -np.pi/2, 
                 0, np.pi/2, np.pi/4, 0.06, 0.06]]*2
    action2 = [[0.0, -np.pi/4, 0.0, -np.pi/2, 
                 0, np.pi/2, np.pi/4, 0.06, 0.06]]*8
    
    action = action1 + action2
    action = torch.tensor(action, dtype=torch.float32)

    return action



In [4]:
# Define CMA-ES parameters
num_guess = 5
popsize = 100
num_iterations = 10
# n_envs = num_guess * popsize
max_batch = MAX_BATCH
K = 2

# Define p2p motion
T = 1
dt = 0.01
n_joints = 7
basis = 'bs'
bounds = None
p2p_motion = Point2PointMotion(T=T, dt=dt, K=K, basis=basis, n=n_joints,
                            bounds=bounds, device=device)


In [5]:
# Generate the environment information
env_infos = env_infos_gen()

In [6]:
import os
if os.path.exists("mp_traj_configuration.npy"):
    theta_t = np.load("mp_traj_configuration.npy")
else:
    raise ValueError("MP trajectory file not found")

num_res = 5
theta_t = theta_t[:num_res]

# Define CMA-ES parameters
env = PandaShelf(n_envs=theta_t.shape[0], gpu=True, use_gui=True,
        p2p_motion=p2p_motion)
env.load_world(env_infos, render_utility=True)

env.reset(env_infos)

robot = env_infos["robot"]["id"]
eef = robot.get_link("grasping_target_hand")
eef_pos = eef.get_pos()
# print("eef position", eef_pos)
env.cam.start_recording()
env.forward_configuration(env_infos, theta_t, record=True)
env.cam.stop_recording(save_to_filename="mp_video.mp4", fps=20)

eef_pos = eef.get_pos()
print("eef position", eef_pos)

robot_confs = robot.get_qpos()
print("robot configuration", robot_confs)


[Genesis] [10:42:09] [WARNING] Overriding base link's pos with user provided value in morph.
[Genesis] [10:42:09] [WARNING] Overriding base link's quat with user provided value in morph.


eef position tensor([[0.5988, 0.1505, 1.0984],
        [0.5999, 0.1503, 1.1010],
        [0.5992, 0.1488, 1.0998],
        [0.6011, 0.1474, 1.1000],
        [0.5992, 0.1451, 1.0991]], device='cuda:0', dtype=torch.float32)
robot configuration tensor([[ 3.9883e+00, -6.5690e-01, -4.0313e+00, -2.3078e+00, -5.0365e-01,
          4.4269e+00, -1.9408e+00,  4.0743e-02, -7.4269e-04],
        [ 3.9899e+00, -6.5542e-01, -4.0339e+00, -2.3049e+00, -5.0054e-01,
          4.4265e+00, -1.9440e+00,  4.0733e-02, -7.3250e-04],
        [ 3.9895e+00, -6.5565e-01, -4.0369e+00, -2.3078e+00, -5.0598e-01,
          4.4263e+00, -1.9427e+00,  4.0756e-02, -7.5608e-04],
        [ 3.9846e+00, -6.5710e-01, -4.0355e+00, -2.3025e+00, -5.0661e-01,
          4.4232e+00, -1.9414e+00,  4.0692e-02, -6.9202e-04],
        [ 3.9865e+00, -6.5354e-01, -4.0372e+00, -2.3130e+00, -5.0271e-01,
          4.4241e+00, -1.9378e+00,  4.0702e-02, -7.0192e-04]], device='cuda:0',
       dtype=torch.float32)
